In [2]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
import os

np.random.seed(0)
os.makedirs("/home/claude/figs2", exist_ok=True)

BLUE = "#4C72B0"
RED  = "#DD4444"

# ── 1. Dataset ────────────────────────────────────────────────
data, _ = make_moons(n_samples=5000, noise=0.06)
data = data.astype(np.float32)
# centre & scale to roughly [-1.5, 1.5]
data -= data.mean(axis=0)
data /= data.std()

# ── 2. Noise schedule ─────────────────────────────────────────
T         = 300
betas     = np.linspace(1e-4, 0.02, T, dtype=np.float32)
alphas    = 1.0 - betas
alpha_bar = np.cumprod(alphas)
sqrt_ab   = np.sqrt(alpha_bar)
sqrt_1mab = np.sqrt(1.0 - alpha_bar)

def q_sample(x0, t):
    eps = np.random.randn(*x0.shape).astype(np.float32)
    return sqrt_ab[t, None] * x0 + sqrt_1mab[t, None] * eps, eps

# ── 3. Time embedding ─────────────────────────────────────────
EMB = 32
def time_emb(t_batch):
    half  = EMB // 2
    freqs = np.exp(-np.log(10000) * np.arange(half) / half).astype(np.float32)
    args  = t_batch[:, None] * freqs[None, :]
    return np.concatenate([np.sin(args), np.cos(args)], axis=1)

# ── 4. MLP (NumPy) ────────────────────────────────────────────
# Input: [x_t(2) || t_emb(32)] = 34-D
# Hidden: 128 → 128
# Output: 2
DIMS = [2 + EMB, 128, 128, 2]

def init():
    P = []
    for i in range(len(DIMS)-1):
        W = np.random.randn(DIMS[i], DIMS[i+1]).astype(np.float32) * np.sqrt(2/DIMS[i])
        b = np.zeros(DIMS[i+1], dtype=np.float32)
        P.append([W, b])
    return P

def forward(P, x):
    cache, h = [], x
    for i, (W, b) in enumerate(P):
        z = h @ W + b
        a = np.maximum(0, z) if i < len(P)-1 else z
        cache.append((h, z))
        h = a
    return h, cache

def backward(P, cache, g):
    grads = []
    for i in reversed(range(len(P))):
        h, z = cache[i]
        if i < len(P)-1:
            g = g * (z > 0)
        grads.insert(0, [h.T @ g, g.sum(0)])
        g = g @ P[i][0].T
    return grads

# Adam
def make_adam(P):
    return [[np.zeros_like(W), np.zeros_like(b)] for W, b in P], \
           [[np.zeros_like(W), np.zeros_like(b)] for W, b in P]

def adam(P, G, M, V, lr, step, b1=0.9, b2=0.999, eps=1e-8):
    for i, ((W, b), (dW, db), (mW, mb), (vW, vb)) in enumerate(zip(P, G, M, V)):
        for arr, d, m, v, idx in [(W, dW, mW, vW, 0), (b, db, mb, vb, 1)]:
            m[:] = b1*m + (1-b1)*d
            v[:] = b2*v + (1-b2)*d**2
            P[i][idx] -= lr * (m/(1-b1**step)) / (np.sqrt(v/(1-b2**step)) + eps)
    return P, M, V

# ── 5. Training ───────────────────────────────────────────────
EPOCHS, BS, LR = 2500, 256, 3e-4
P = init()
M, V = make_adam(P)
losses = []
step = 0

print("Training …")
for ep in range(EPOCHS):
    idx = np.random.permutation(len(data))
    ep_loss, n = 0.0, 0
    for s in range(0, len(data)-BS, BS):
        x0 = data[idx[s:s+BS]]
        t  = np.random.randint(0, T, BS)
        xt, eps = q_sample(x0, t)
        x_in = np.concatenate([xt, time_emb(t.astype(np.float32))], axis=1)
        pred, cache = forward(P, x_in)
        diff = pred - eps
        loss = (diff**2).mean()
        g    = 2*diff / (BS*2)
        grads = backward(P, cache, g)
        step += 1
        P, M, V = adam(P, grads, M, V, LR, step)
        ep_loss += loss; n += 1
    losses.append(ep_loss/n)
    if (ep+1) % 500 == 0:
        print(f"  Epoch {ep+1}/{EPOCHS}  loss={losses[-1]:.4f}")

print("Done.")

# ── 6. Sampling ───────────────────────────────────────────────
def sample(n=800):
    x = np.random.randn(n, 2).astype(np.float32)
    snaps = {}
    for t in reversed(range(T)):
        ta  = np.full(n, t, dtype=np.float32)
        xin = np.concatenate([x, time_emb(ta)], axis=1)
        eps_hat, _ = forward(P, xin)
        coef = betas[t] / sqrt_1mab[t]
        mean = (x - coef*eps_hat) / np.sqrt(alphas[t])
        x    = mean + (np.sqrt(betas[t]) * np.random.randn(*x.shape) if t>0 else 0)
        if t in {250, 180, 100, 50, 10, 0}:
            snaps[t] = x.copy()
    return x, snaps

print("Sampling …")
generated, snaps = sample(1000)

# ── 7. Figures ────────────────────────────────────────────────

# Fig A – Training loss
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(losses, color=BLUE, lw=1.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
ax.set_title("Training Loss", fontweight="bold")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("/home/claude/figs2/loss.png", dpi=150); plt.close()

# Fig B – Forward noising (5 snapshots)
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, t in zip(axes, [0, 60, 130, 200, 299]):
    xt, _ = q_sample(data[:500], np.full(500, t))
    ax.scatter(xt[:,0], xt[:,1], s=5, alpha=0.5, c=BLUE)
    ax.set_title(f"t = {t}", fontsize=11, fontweight="bold")
    ax.set_xlim(-3,3); ax.set_ylim(-3,3); ax.set_aspect("equal"); ax.grid(alpha=0.2)
    ax.tick_params(labelsize=7)
fig.suptitle("Forward Process: Adding Noise  q(xₜ | x₀)", fontsize=12, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig("/home/claude/figs2/forward.png", dpi=150, bbox_inches="tight"); plt.close()

# Fig C – Reverse denoising (6 snapshots)
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, t in zip(axes, sorted(snaps.keys(), reverse=True)):
    pts = snaps[t][:400]
    ax.scatter(pts[:,0], pts[:,1], s=5, alpha=0.5, c=RED)
    ax.set_title(f"t = {t}", fontsize=11, fontweight="bold")
    ax.set_xlim(-3,3); ax.set_ylim(-3,3); ax.set_aspect("equal"); ax.grid(alpha=0.2)
    ax.tick_params(labelsize=7)
fig.suptitle("Reverse Process: Removing Noise  p_θ(x_{t-1} | xₜ)", fontsize=12, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig("/home/claude/figs2/reverse.png", dpi=150, bbox_inches="tight"); plt.close()

# Fig D – Real vs Generated
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 4))
a1.scatter(data[:600,0], data[:600,1], s=8, alpha=0.5, c=BLUE)
a1.set_title("Real Data (Two Moons)", fontweight="bold", fontsize=12)
a1.set_aspect("equal"); a1.grid(alpha=0.2); a1.set_xlim(-3,3); a1.set_ylim(-3,3)

a2.scatter(generated[:600,0], generated[:600,1], s=8, alpha=0.5, c=RED)
a2.set_title("Generated Samples (DDPM)", fontweight="bold", fontsize=12)
a2.set_aspect("equal"); a2.grid(alpha=0.2); a2.set_xlim(-3,3); a2.set_ylim(-3,3)
fig.tight_layout()
fig.savefig("/home/claude/figs2/compare.png", dpi=150); plt.close()

print("Figures saved.")

Training …
  Epoch 500/2500  loss=0.4265
  Epoch 1000/2500  loss=0.4269
  Epoch 1500/2500  loss=0.4120
  Epoch 2000/2500  loss=0.4093
  Epoch 2500/2500  loss=0.3938
Done.
Sampling …
Figures saved.
